| [0.0 Data Collection](00_data_collection.ipynb) | 1.0 Preprocessing | [2.0 Spatial Autocorrelation](02_spatial_autocorrelation.ipynb) | [3.0 MGWR](03_mgwr_analysis.ipynb) | [4.0 ML Classification](04_ml_classification.ipynb) |

# 1.0 | Data Preprocessing & Feature Engineering

This notebook constructs the MSOA-level analysis dataset by:
1. Cleaning STATS19 collision records (2021-2024)
2. Spatial joining accident points to MSOA polygons
3. Aggregating IMD 2019 deprivation scores from LSOA to MSOA
4. Computing road network metrics (road density, junction density)
5. Merging all features into a single GeoDataFrame

**Output**: `data/processed/msoa_analysis.gpkg` (combined 2021-2024) + yearly files

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import (
    MSOA_ANALYSIS_GPKG, PROCESSED_DIR, RAW_DIR, YEARS,
    FIGURES_DIR, FIGURE_DPI
)

plt.rcParams.update({
    'font.family': 'serif',
    'figure.dpi': 150,
    'axes.grid': True,
    'grid.alpha': 0.3
})

## 1.1 | Build Complete Feature Dataset

The `build_features()` function orchestrates the entire preprocessing pipeline.  
Set `force=True` to rebuild from scratch; otherwise cached results are loaded.

In [ ]:
from src.analysis.preprocessing.feature_builder import build_features

gdf = build_features(force=False)
print(f"Dataset shape: {gdf.shape}")
print(f"CRS: {gdf.crs}")
print(f"Memory usage: {gdf.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## 1.2 | Understanding the Data

Before proceeding to analysis, the structure, types and completeness of the dataset need to be assessed.  
The table below shows all columns and their data types.

In [ ]:
print(f"Number of MSOAs: {len(gdf):,}")
print(f"Number of columns: {len(gdf.columns)}")
print(f"\nColumn types:")
print("─" * 50)
for col in gdf.columns:
    dtype = str(gdf[col].dtype)
    null_count = gdf[col].isnull().sum()
    null_pct = f"({null_count / len(gdf) * 100:.1f}% missing)" if null_count > 0 else ""
    print(f"  {col:30s} {dtype:15s} {null_pct}")

In [ ]:
gdf.head()

### Missing values

Checking for missing values across all columns. Columns with no missing values are omitted.

In [ ]:
missing = gdf.isnull().sum()
missing_pct = (missing / len(gdf) * 100).round(2)
missing_report = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
})
missing_report = missing_report[missing_report['missing_count'] > 0]

if len(missing_report) == 0:
    print("No missing values found across any column.")
else:
    print(f"Columns with missing values: {len(missing_report)}")
    missing_report.sort_values('missing_pct', ascending=False)

### Summary statistics

Descriptive statistics for all numerical columns. Large standard deviations relative to means  
indicate substantial spatial variation across MSOAs.

In [ ]:
gdf.describe()

### 1.2 | Figure 1 | Distribution of key variables

Histograms of the main analysis variables to assess distributional properties.

In [ ]:
plot_cols = [
    'accident_rate_per_10k', 'imd_score', 'road_density',
    'junction_density', 'urban_pct', 'wet_road_pct', 'dark_pct'
]
available_cols = [c for c in plot_cols if c in gdf.columns]

n_cols = 4
n_rows = int(np.ceil(len(available_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(available_cols):
    gdf[col].hist(bins=50, ax=axes[i], edgecolor='white', alpha=0.8)
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=11)
    axes[i].set_ylabel('Frequency')

# Hide unused subplots
for j in range(len(available_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution of Key Variables across MSOAs', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

### 1.2 | Figure 2 | Correlation matrix

Pairwise Pearson correlations between all numerical features.  
Strong correlations (|r| > 0.7) may indicate multicollinearity concerns for regression models.

In [ ]:
numeric_cols = gdf.select_dtypes(include=[np.number]).columns.tolist()
# Remove geometry-related if present
numeric_cols = [c for c in numeric_cols if c not in ['OBJECTID']]

corr = gdf[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix of MSOA Features', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

### 1.2 | Figure 3 | Spatial distribution of accident rate and deprivation

Choropleth maps showing how accident rate and IMD deprivation vary across England.  
Visual inspection suggests potential spatial clustering — formally tested in the next notebook.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 10))

gdf.plot(column='accident_rate_per_10k', cmap='YlOrRd', legend=True, ax=axes[0],
         edgecolor='face', linewidth=0.1,
         legend_kwds={'label': 'Rate per 10,000', 'shrink': 0.6})
axes[0].set_title('Accident Rate per 10,000 Population', fontweight='bold')
axes[0].axis('off')

gdf.plot(column='imd_score', cmap='RdPu', legend=True, ax=axes[1],
         edgecolor='face', linewidth=0.1,
         legend_kwds={'label': 'IMD Score', 'shrink': 0.6})
axes[1].set_title('IMD 2019 Deprivation Score', fontweight='bold')
axes[1].axis('off')

plt.suptitle('Spatial Distribution of Key Variables', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

### 1.2 | Figure 4 | Log-transformed accident rate

The raw accident rate is right-skewed. A log transformation is applied for regression modelling  
to better satisfy normality assumptions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

gdf['accident_rate_per_10k'].hist(bins=50, ax=axes[0], edgecolor='white')
axes[0].set_title('Accident Rate (Raw)', fontweight='bold')
axes[0].set_xlabel('Rate per 10,000')
axes[0].set_ylabel('Frequency')

if 'log_accident_rate' in gdf.columns:
    gdf['log_accident_rate'].hist(bins=50, ax=axes[1], edgecolor='white', color='coral')
    axes[1].set_title('Accident Rate (Log-transformed)', fontweight='bold')
    axes[1].set_xlabel('Log Rate')
    axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 1.3 | Yearly Comparison

To assess temporal stability, yearly MSOA files are also generated.  
Note that 2021 represents a post-COVID recovery period with potentially atypical patterns.

In [ ]:
print("Yearly MSOA datasets:")
print("─" * 60)
yearly_stats = []

for year in YEARS:
    path = PROCESSED_DIR / f"msoa_yearly_{year}.gpkg"
    if path.exists():
        y = gpd.read_file(path)
        stats = {
            'Year': year,
            'MSOAs': len(y),
            'Mean Rate': y['accident_rate_per_10k'].mean(),
            'Median Rate': y['accident_rate_per_10k'].median(),
            'Std Rate': y['accident_rate_per_10k'].std()
        }
        yearly_stats.append(stats)
        print(f"  {year}: {len(y):,} MSOAs, "
              f"mean rate = {stats['Mean Rate']:.2f}, "
              f"median = {stats['Median Rate']:.2f}")
    else:
        print(f"  {year}: NOT FOUND")

if yearly_stats:
    pd.DataFrame(yearly_stats).set_index('Year')

## 1.4 | Summary

The preprocessing pipeline has produced a complete MSOA-level dataset with the following key features:

| Variable | Description | Source |
|----------|-------------|--------|
| `accident_rate_per_10k` | Collision rate per 10,000 population (2021-2024) | STATS19 + ONS |
| `imd_score` | Population-weighted mean IMD 2019 score | MHCLG |
| `road_density` | Road km per sq km | OS Open Roads |
| `junction_density` | Junctions per sq km | OS Open Roads |
| `urban_pct` | % urban collisions | STATS19 |
| `wet_road_pct` | % wet road surface collisions | STATS19 |
| `dark_pct` | % collisions in darkness | STATS19 |

Proceed to **Notebook 02** for spatial autocorrelation analysis.

---
| [0.0 Data Collection](00_data_collection.ipynb) | 1.0 Preprocessing | [2.0 Spatial Autocorrelation](02_spatial_autocorrelation.ipynb) | [3.0 MGWR](03_mgwr_analysis.ipynb) | [4.0 ML Classification](04_ml_classification.ipynb) |